In [ ]:
pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 6.5 MB/s eta 0:00:00




### **Utilize Nvidia GPU**




In [ ]:
!nvidia-smi

Wed Sep  9 08:38:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# **Loading Dataset**

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="sw4qEJVULbJ77VNSAXor")
project = rf.workspace("treedataset-clsqo").project("tree-top-view")
version = project.version(6)
dataset = version.download("yolov8")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 144.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.1
    Uninstalling typer-0.27.1:
      Successfully uninstalled typer-0.27.1
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Tree-Top-View-6 in yolov8:: 100%|██████████| 2780/2780 [00:00<00:00, 7437.35it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


### **Fine-Tuning the YOLOv26 Model**

In [13]:
from ultralytics import YOLO

# Load a pre-trained YOLOv8 model
model = YOLO('yolov8m.pt')

# Fine-tune the model exclusively on your agroforestry dataset
results = model.train(
    data='/content/Tree-Top-View-6/data.yaml',
    epochs=100,           # Adjust based on dataset size and convergence
    imgsz=640,            # Image resolution for training
    batch=16,             # Batch size (reduce if GPU memory is limited)
    device=0,             # Set to 0 for GPU, 'cpu' for CPU
    project='Agroforestry',
    name='Tree_Detection_V1'
)

Ultralytics 8.4.146 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Tree-Top-View-6/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Tree_Detection_V1-2, nbs=64, nms=None, o

KeyboardInterrupt: 

### Inference, NMS Tuning, and Counting

In [3]:
from ultralytics import YOLO

# Load your newly fine-tuned weights
best_model = YOLO('/content/runs/detect/Agroforestry/Tree_Detection_V1/weights/best.pt')

# Run inference on a test image with tuned NMS thresholds
results = best_model.predict(
    source='/content/tree1.jpg',
    conf=0.45,  # Confidence threshold: Discard predictions below 45% certainty
    iou=0.4,    # NMS IoU threshold: Adjust this to balance dense overlaps vs. double counting
    save=True   # Save the output image with bounding boxes drawn
)

# Extract required outputs: Locations, Total Counts, and Confidence Scores
for result in results:
    boxes = result.boxes

    # 1. Total Tree Count
    total_trees = len(boxes)
    print(f"Total Trees Detected: {total_trees}\n")

    # 2. Individual Locations and Confidence Scores
    for i, box in enumerate(boxes):
        # Coordinates in [x1, y1, x2, y2] pixel format
        coords = box.xyxy[0].tolist()
        # Confidence score for the detection
        conf = box.conf[0].item()

        print(f"Tree {i+1}:")
        print(f"  Location: [X1: {coords[0]:.1f}, Y1: {coords[1]:.1f}, X2: {coords[2]:.1f}, Y2: {coords[3]:.1f}]")
        print(f"  Confidence: {conf:.2f}")


image 1/1 /content/tree1.jpg: 640x384 37 tree-tops, 25.0ms
Speed: 76.2ms preprocess, 25.0ms inference, 37.5ms postprocess per image at shape (1, 3, 640, 384)
Results saved to /content/runs/detect/predict-3
Total Trees Detected: 37

Tree 1:
  Location: [X1: 325.8, Y1: 690.3, X2: 493.4, Y2: 838.4]
  Confidence: 0.90
Tree 2:
  Location: [X1: 652.2, Y1: 1042.6, X2: 736.0, Y2: 1145.7]
  Confidence: 0.79
Tree 3:
  Location: [X1: 328.6, Y1: 1191.9, X2: 455.7, Y2: 1306.6]
  Confidence: 0.78
Tree 4:
  Location: [X1: 6.0, Y1: 810.8, X2: 143.1, Y2: 946.0]
  Confidence: 0.75
Tree 5:
  Location: [X1: 153.1, Y1: 770.3, X2: 291.3, Y2: 886.9]
  Confidence: 0.75
Tree 6:
  Location: [X1: 468.8, Y1: 729.3, X2: 624.9, Y2: 877.4]
  Confidence: 0.74
Tree 7:
  Location: [X1: 60.0, Y1: 1184.7, X2: 155.8, Y2: 1305.7]
  Confidence: 0.74
Tree 8:
  Location: [X1: 657.8, Y1: 706.1, X2: 733.7, Y2: 796.2]
  Confidence: 0.73
Tree 9:
  Location: [X1: 496.9, Y1: 870.6, X2: 650.4, Y2: 1021.0]
  Confidence: 0.73
Tree 10